# logit-thermo A2-7c：本滴定のやり直しの分析（CPU）

- **必要な secret：なし**。**ランタイム：CPU（標準）**。見込みは 5〜10 分。**データ集めが終わってから**回してください。
- `A2_7C/` の Qwen の 8 本の腕を読み、A2-7b と同じ決め方で判定して `A2_7C/a2_7c_summary.json` に書きます。**結果は表示しません。**
- 計算のコード：A2-2・A2-4・A2-7b の分析のコードは Drive の前のノートブックから取り出してハッシュで確かめ、`scripts/a2_7c_analyze.py`（`b95d4f5f4aa8180c…`）は埋め込み（データを開く前に固定）。
- **通知**：🚀・🎉・⚠️。名札は「logit-thermo A2-7c 本滴定やり直し 分析（Qwen の判定、CPU）」。終わったら、またはエラーで止まったら、ランタイムを切ります。


In [ ]:
# セル1：通知、GPU、Drive
import os, time, urllib.request
NTFY_TOPIC = 'YOUR_NTFY_TOPIC'
NB_LABEL = 'logit-thermo A2-7c 本滴定やり直し 分析（Qwen の判定、CPU）'
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=(msg + '\n[' + NB_LABEL + ']').encode('utf-8'),
                                     headers={'Title': 'logit-thermo - A2-7c analyze', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
_NOTIFIED = {'error': False}
def _on_error(result):
    if result.error_in_exec is not None and not _NOTIFIED['error']:
        _NOTIFIED['error'] = True
        import traceback
        e = result.error_in_exec
        txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
        v = os.environ.get('HF_TOKEN')
        if v: txt = txt.replace(v, '***')
        txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
        notify('⚠️ 止まりました（エラー）：\n' + txt, tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('🚀 スタートしました（これが届けば通知の設定は OK です）', tags='rocket')

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# セル2：分析のコード
import ast, glob, hashlib, json
def take_src(want):
    """want：{変数の名前: sha256}。Drive の logit-thermo のノートブックから、埋め込んだコードを取り出してハッシュで確かめる。"""
    got = {}
    for p in sorted(glob.glob('/content/drive/MyDrive/logit-thermo' + '/**/*.ipynb', recursive=True)):
        try:
            cells = json.load(open(p))['cells']
        except Exception:
            continue
        for c in cells:
            s = c['source'] if isinstance(c['source'], str) else ''.join(c['source'])
            if c.get('cell_type') != 'code' or 'SRC' not in s:
                continue
            try:
                body = ast.parse(s).body
            except SyntaxError:
                continue
            for n in body:
                if isinstance(n, ast.Assign) and isinstance(n.targets[0], ast.Name) and n.targets[0].id in want and n.targets[0].id not in got:
                    v = ast.literal_eval(n.value)
                    if hashlib.sha256(v.encode()).hexdigest() == want[n.targets[0].id]:
                        got[n.targets[0].id] = v
        if len(got) == len(want):
            break
    miss = [k for k in want if k not in got]
    assert not miss, ('Drive のノートブックからコードを取り出せませんでした（前のノートブックを動かしていませんか）：', miss)
    return got
_s = take_src({'SRCA22': 'fe1f96db1f7e84031e85a70bcc8dcbb95c9fc06b3367657ff4074a9f9d1343f7', 'SRCA24': 'e70bad0db9b7780c2059aae52ad06debe87757bc582791deefb58c5f435f84a6', 'SRC7BA': 'd4da9ac740e2b6451845da8496ff90e451d5ba11bbcef28e1206da2cc61d1135'})
SRCA22, SRCA24, SRC7BA = _s['SRCA22'], _s['SRCA24'], _s['SRC7BA']
SRC7CA = '"""A2-7c（ゲート docs/A2_7C_GATE.md）の分析。scripts/a2_7b_analyze.py を実行し、その自己テストを通したあとの同じ名前空間で実行して、\n温度の並びとモデルだけを置きかえる（判定の決め方は A2-7b のまま。データを開く前に固定する）。\n"""\nB7A_TEMPS = [0.6, 0.7, 0.8, 0.85, 0.9, 0.95, 1.0, 1.05]\nB7A_MODELS = [\'qwen2.5-1.5b\']\n\n\ndef analyze7c(d):\n    r = analyze7b(d)\n    m = r[\'models\'][\'qwen2.5-1.5b\']\n    r[\'CQ1\'] = m[\'BQ1\'][\'word\']\n    r[\'CQ2\'] = m[\'BQ2_word\']\n    return r\n'
assert hashlib.sha256(SRC7CA.encode()).hexdigest() == 'b95d4f5f4aa8180c132ca85071a98b4eb605060c950ee19737d123e916dcb4ce'
exec(SRCA22)
exec(SRCA24)
exec(SRC7BA)
print('自己テスト（A2-7b の分析）：', self_test_a27b_analyze())
exec(SRC7CA)
assert B7A_TEMPS == [0.6, 0.7, 0.8, 0.85, 0.9, 0.95, 1.0, 1.05] and B7A_MODELS == ['qwen2.5-1.5b']
print('温度とモデルを置きかえました')


In [ ]:
# セル3：分析して保存（結果は表示しない）
import json
try:
    r = analyze7c('/content/drive/MyDrive/logit-thermo/A2_7C')
    r['a2_7c_analyze_sha256'] = 'b95d4f5f4aa8180c132ca85071a98b4eb605060c950ee19737d123e916dcb4ce'
    json.dump(_clean(r), open('/content/drive/MyDrive/logit-thermo/A2_7C' + '/a2_7c_summary.json', 'w'), ensure_ascii=False, separators=(',', ':'))
    print('保存しました')
except Exception as e:
    import traceback
    _NOTIFIED['error'] = True
    txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
    v = os.environ.get('HF_TOKEN')
    if v: txt = txt.replace(v, '***')
    txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
    notify('⚠️ A2-7c の分析で止まりました。ランタイムを切ります：\n' + txt, tags='warning')
    if globals().get('AUTO_DISCONNECT', True):
        from google.colab import drive as _d, runtime as _r
        _d.flush_and_unmount(); time.sleep(3); _r.unassign()
    raise


In [ ]:
# 最後：終わりの通知と、ランタイムを切る
notify('🎉 終わりました：判定を A2_7C/a2_7c_summary.json に保存しました', tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount(); time.sleep(3); _runtime.unassign()
